# TP 2 — Classification MNIST avec un CNN (Solution) ✅

**Fondations Deep Learning | Papa Séga WADE**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.manifold import TSNE

# 1. Données
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = np.expand_dims(X_train.astype('float32') / 255.0, -1)
X_test  = np.expand_dims(X_test.astype('float32') / 255.0, -1)
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test, 10)

# 2. Modèle CNN avec couche d'embeddings
inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)
x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)
x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.Flatten()(x)
embeddings = layers.Dense(128, activation='relu', name='embeddings')(x)
x = layers.Dropout(0.5)(embeddings)
outputs = layers.Dense(10, activation='softmax')(x)
model = Model(inputs=inputs, outputs=outputs)

# 3. Entraijnement
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train_cat, epochs=10, batch_size=128, validation_split=0.1, verbose=1)

test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"🏆 Test Accuracy : {test_acc*100:.2f}%")

In [ ]:
# Embeddings + t-SNE
embedding_model = Model(inputs=model.input, outputs=model.get_layer('embeddings').output)
embeds = embedding_model.predict(X_test[:2000], verbose=0)
embeds_2d = TSNE(n_components=2, random_state=42).fit_transform(embeds)

colors = plt.cm.tab10(np.linspace(0, 1, 10))
fig, ax = plt.subplots(figsize=(12, 10))
for digit in range(10):
    mask = y_test[:2000] == digit
    ax.scatter(embeds_2d[mask, 0], embeds_2d[mask, 1],
               c=[colors[digit]], label=str(digit), s=15, alpha=0.7)
ax.legend(title='Chiffre')
ax.set_title('t-SNE des Embeddings CNN MNIST')
plt.tight_layout()
plt.show()